# Explainable Multi-Crop Leaf Disease Classification Using Custom CNN and Transfer Learning: A Tomato Leaf Disease Case Study
Multi-crop leaf disease classification methodology demonstrated on tomato (10 classes); custom CNN + transfer learning (MobileNetV2, EfficientNetB0).

## 1. Setup & Imports

In [ ]:
import os
import glob
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

## 2. Dataset Path Detection

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
MAX_SEARCH_DEPTH = 6
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
IMAGE_EXTENSIONS = IMAGE_EXT
SPLIT_NAMES = {"train", "valid", "val", "validation", "test"}
VALID_FOLDER_ALIASES = {"valid", "val", "validation"}
ROOT_EXCLUDE_NAMES = {"kaggle", "datasets", "notebooks", "input"}
EXPECTED_NUM_CLASSES = 10


def _is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXT


def _dir_contains_images(directory: Path) -> bool:
    if not directory.is_dir():
        return False
    try:
        for child in directory.iterdir():
            if _is_image_file(child):
                return True
    except (OSError, PermissionError):
        return False
    return False


def _count_class_folders(directory: Path) -> int:
    if not directory.is_dir():
        return 0
    count = 0
    try:
        for child in directory.iterdir():
            if child.is_dir() and _dir_contains_images(child):
                count += 1
    except (OSError, PermissionError):
        pass
    return count


def _split_subfolder_has_class_images(split_dir: Path) -> bool:
    if not split_dir.is_dir():
        return False
    try:
        for child in split_dir.iterdir():
            if child.is_dir() and _dir_contains_images(child):
                return True
    except (OSError, PermissionError):
        return False
    return False


def _find_split_subdirs(root: Path) -> dict:
    mapping = {}
    try:
        for child in root.iterdir():
            if not child.is_dir():
                continue
            name = child.name.lower()
            if name not in SPLIT_NAMES:
                continue
            if not _split_subfolder_has_class_images(child):
                continue
            if name == "train":
                mapping["train"] = child
            elif name in VALID_FOLDER_ALIASES:
                mapping["valid"] = child
            elif name == "test":
                mapping["test"] = child
    except (OSError, PermissionError):
        pass
    return mapping


def _depth_from_input(path: Path, input_root: Path) -> int:
    try:
        return len(path.relative_to(input_root).parts)
    except ValueError:
        return 0


def _score_candidate(directory: Path) -> tuple:
    if directory.name.lower() in ROOT_EXCLUDE_NAMES:
        return 0, None, 0

    split_map = _find_split_subdirs(directory)
    if split_map and "train" in split_map:
        n_class = _count_class_folders(split_map["train"])
        if n_class >= 2:
            return 100 + len(split_map), "split", n_class

    n_class = _count_class_folders(directory)
    if n_class >= 2:
        return n_class, "flat", n_class

    return 0, None, 0


def _walk_candidates(input_root: Path, max_depth: int = MAX_SEARCH_DEPTH) -> list:
    candidates = []

    def _walk(current: Path, depth: int) -> None:
        if depth > max_depth:
            return
        candidates.append(current)
        if depth == max_depth:
            return
        try:
            for child in sorted(current.iterdir(), key=lambda p: p.name.lower()):
                if child.is_dir():
                    _walk(child, depth + 1)
        except (OSError, PermissionError):
            pass

    if not input_root.exists():
        raise FileNotFoundError(
            f"Kaggle input path not found: {input_root}. Attach the tomato leaf disease dataset."
        )

    try:
        for child in sorted(input_root.iterdir(), key=lambda p: p.name.lower()):
            if child.is_dir():
                _walk(child, 1)
    except (OSError, PermissionError):
        pass

    return candidates


def _find_dataset_root(input_root: Path) -> tuple:
    candidates = _walk_candidates(input_root)
    scored = []
    for candidate in candidates:
        score, layout, n_class = _score_candidate(candidate)
        if score > 0 and layout:
            depth = _depth_from_input(candidate, input_root)
            scored.append((candidate, depth, layout, score, n_class))

    print(f"--- Dataset root candidates (depth <= {MAX_SEARCH_DEPTH}) ---")
    for path, depth, layout, score, n_class in sorted(
        scored, key=lambda x: (-x[3], x[1], str(x[0]).lower())
    ):
        print(
            f"  {path} | depth={depth} | layout={layout} | "
            f"score={score} | n_class_folders={n_class}"
        )

    if not scored:
        raise FileNotFoundError(
            "Could not detect any image class folders under /kaggle/input. "
            "Check that the correct tomato leaf disease dataset is attached."
        )

    scored.sort(key=lambda x: (-x[3], x[1], str(x[0]).lower()))
    chosen, _, chosen_layout, _, _ = scored[0]
    print(f"\nChosen DATA_ROOT: {chosen}")
    return chosen, chosen_layout


def _get_class_names_from_dir(directory: Path) -> list:
    return [
        d.name
        for d in sorted(directory.iterdir(), key=lambda p: (p.name.lower(), p.name))
        if d.is_dir() and _dir_contains_images(d)
    ]


DATA_ROOT_PATH, LAYOUT = _find_dataset_root(INPUT_ROOT)
DATA_ROOT = str(DATA_ROOT_PATH)

TRAIN_DIR = VALID_DIR = TEST_DIR = None

if LAYOUT == "split":
    split_map = _find_split_subdirs(DATA_ROOT_PATH)
    TRAIN_DIR = str(split_map["train"]) if split_map.get("train") else None
    VALID_DIR = str(split_map["valid"]) if split_map.get("valid") else None
    TEST_DIR = str(split_map["test"]) if split_map.get("test") else None

    if TRAIN_DIR is None:
        raise FileNotFoundError(
            f"Split layout detected at {DATA_ROOT}, but no 'train' folder was found."
        )
elif LAYOUT == "flat":
    if _count_class_folders(DATA_ROOT_PATH) < 2:
        raise FileNotFoundError(
            f"Flat layout detected at {DATA_ROOT}, but fewer than 2 class folders were found."
        )
else:
    raise FileNotFoundError(
        f"Dataset root found at {DATA_ROOT}, but layout could not be determined."
    )

if LAYOUT == "split":
    class_names = _get_class_names_from_dir(Path(TRAIN_DIR))
else:
    class_names = _get_class_names_from_dir(DATA_ROOT_PATH)

NUM_CLASSES = len(class_names)

if NUM_CLASSES != EXPECTED_NUM_CLASSES:
    print(
        f"NOTE: Detected {NUM_CLASSES} classes from train folders; "
        f"{EXPECTED_NUM_CLASSES} expected for this project."
    )

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"LAYOUT: {LAYOUT}")
print(f"TRAIN_DIR: {TRAIN_DIR}")
print(f"VALID_DIR: {VALID_DIR}")
print(f"TEST_DIR: {TEST_DIR}")
print(f"NUM_CLASSES: {NUM_CLASSES}")

## 3. Class Detection & Image Counts

In [ ]:
def _count_images_in_dir(directory: Path) -> int:
    if directory is None or not directory.exists():
        return 0
    return sum(1 for f in directory.iterdir() if _is_image_file(f))


def _get_class_names_from_dir(directory: Path) -> list:
    return [
        d.name
        for d in sorted(directory.iterdir(), key=lambda p: (p.name.lower(), p.name))
        if d.is_dir() and _dir_contains_images(d)
    ]


if LAYOUT == "split":
    class_names = _get_class_names_from_dir(Path(TRAIN_DIR))
elif LAYOUT == "flat":
    class_names = _get_class_names_from_dir(DATA_ROOT_PATH)
else:
    class_names = []

NUM_CLASSES = len(class_names)

records = []
for class_name in class_names:
    if LAYOUT == "split":
        train_count = _count_images_in_dir(Path(TRAIN_DIR) / class_name)
        valid_count = _count_images_in_dir(Path(VALID_DIR) / class_name) if VALID_DIR else 0
        test_count = _count_images_in_dir(Path(TEST_DIR) / class_name) if TEST_DIR else 0
        total = train_count + valid_count + test_count
        records.append(
            {
                "class": class_name,
                "train": train_count,
                "valid": valid_count,
                "test": test_count,
                "total": total,
            }
        )
    else:
        total = _count_images_in_dir(DATA_ROOT_PATH / class_name)
        records.append({"class": class_name, "total": total})

class_dist_df = pd.DataFrame(records)
print(f"NUM_CLASSES: {NUM_CLASSES}")
print(class_dist_df.to_string(index=False))

class_dist_df.to_csv(f"{RESULTS_DIR}/class_distribution.csv", index=False)
print(f"Saved: {RESULTS_DIR}/class_distribution.csv")

import re


def _normalize_class_name(name: str) -> str:
    s = name.lower().strip()
    s = re.sub(r"_+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


EXPECTED_CLASSES = {
    "Tomato Healthy",
    "Tomato Bacterial Spot",
    "Tomato Early Blight",
    "Tomato Late Blight",
    "Tomato Leaf Mold",
    "Tomato Septoria Leaf Spot",
    "Tomato Spider Mites",
    "Tomato Target Spot",
    "Tomato Yellow Leaf Curl Virus",
    "Tomato Mosaic Virus",
}

expected_normalized = {_normalize_class_name(c) for c in EXPECTED_CLASSES}
detected_normalized = {_normalize_class_name(c) for c in class_names}

print("\n--- Class name sanity check (informational) ---")
print(f"Detected {NUM_CLASSES} classes; expected ~10 tomato classes.")
print("Detected class names:")
for name in class_names:
    print(f"  - {name}")

norm_missing = sorted(expected_normalized - detected_normalized)
norm_extra = sorted(detected_normalized - expected_normalized)
if norm_missing or norm_extra:
    print("\nFYI — normalized name differences (underscore/spacing variants):")
    if norm_missing:
        print(f"  Expected but not detected after normalization: {norm_missing}")
    if norm_extra:
        print(f"  Detected but not in reference set after normalization: {norm_extra}")
else:
    print("\nNormalized names align with the 10 reference tomato classes.")


def _iter_all_image_paths():
    if LAYOUT == "split":
        splits = [("train", TRAIN_DIR)]
        if VALID_DIR:
            splits.append(("valid", VALID_DIR))
        if TEST_DIR:
            splits.append(("test", TEST_DIR))
        for _, split_dir in splits:
            split_path = Path(split_dir)
            for class_name in class_names:
                class_dir = split_path / class_name
                if class_dir.is_dir():
                    for img_path in sorted(class_dir.iterdir()):
                        if _is_image_file(img_path):
                            yield str(img_path)
    else:
        for class_name in class_names:
            class_dir = DATA_ROOT_PATH / class_name
            if class_dir.is_dir():
                for img_path in sorted(class_dir.iterdir()):
                    if _is_image_file(img_path):
                        yield str(img_path)

## 4. Corrupt Image Detection

In [ ]:
corrupt_images = []

for img_path in _iter_all_image_paths():
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception:
        corrupt_images.append(img_path)

print(f"Corrupt/unreadable images found: {len(corrupt_images)}")
if corrupt_images:
    print("Examples (up to 10):")
    for path in corrupt_images[:10]:
        print(f"  {path}")
else:
    print("No corrupt images detected.")

corrupt_out_path = f"{RESULTS_DIR}/corrupt_images.txt"
with open(corrupt_out_path, "w", encoding="utf-8") as f:
    if corrupt_images:
        f.write("\n".join(corrupt_images))
    else:
        f.write("none\n")
print(f"Saved: {corrupt_out_path}")

corrupt_set = set(corrupt_images)
# NOTE: All downstream loaders (Feature 3) must skip any path in corrupt_set.

## 5. Class Distribution Chart

In [ ]:
plot_df = class_dist_df.sort_values("total", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(
    plot_df["class"],
    plot_df["total"],
    color=sns.color_palette("husl", len(plot_df)),
)
ax.set_title("Total Images per Class")
ax.set_xlabel("Class")
ax.set_ylabel("Image Count")
plt.xticks(rotation=45, ha="right")

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
chart_path = f"{RESULTS_DIR}/class_distribution.png"
plt.savefig(chart_path, dpi=150)
plt.show()
print(f"Saved: {chart_path}")

## 6. Sample Images Per Class

In [ ]:
def _readable_samples_for_class(class_name: str, max_samples: int = 2):
    samples = []
    if LAYOUT == "split":
        search_dirs = []
        for split_dir in [TRAIN_DIR, VALID_DIR, TEST_DIR]:
            if split_dir:
                search_dirs.append(Path(split_dir) / class_name)
    else:
        search_dirs = [DATA_ROOT_PATH / class_name]

    for class_dir in search_dirs:
        if not class_dir.is_dir():
            continue
        for img_path in sorted(class_dir.iterdir()):
            if not _is_image_file(img_path):
                continue
            path_str = str(img_path)
            if path_str in corrupt_set:
                continue
            try:
                img = Image.open(path_str).convert("RGB")
                samples.append((path_str, img))
                if len(samples) >= max_samples:
                    return samples
            except Exception:
                continue
    return samples


samples_per_class = {}
for class_name in class_names:
    found = _readable_samples_for_class(class_name, max_samples=1)
    if found:
        samples_per_class[class_name] = found[0][1]

n_classes = len(samples_per_class)
if n_classes == 0:
    print("No readable sample images found to display.")
else:
    ncols = min(5, n_classes)
    nrows = int(np.ceil(n_classes / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (class_name, img) in enumerate(samples_per_class.items()):
        ax = axes[idx]
        ax.imshow(img)
        ax.set_title(class_name, fontsize=9)
        ax.axis("off")

    for idx in range(n_classes, len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    sample_path = f"{RESULTS_DIR}/sample_images.png"
    plt.savefig(sample_path, dpi=150)
    plt.show()
    print(f"Saved: {sample_path}")

## 7. Build File Lists & Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

class_to_index = {class_name: idx for idx, class_name in enumerate(class_names)}
index_to_class = {idx: class_name for class_name, idx in class_to_index.items()}


def _collect_pairs_from_dir(split_dir: str) -> list:
    pairs = []
    split_path = Path(split_dir)
    for class_name in class_names:
        class_dir = split_path / class_name
        if not class_dir.is_dir():
            continue
        label_idx = class_to_index[class_name]
        for img_path in sorted(class_dir.iterdir(), key=lambda p: (p.name.lower(), p.name)):
            if _is_image_file(img_path):
                pairs.append((str(img_path), label_idx))
    return pairs


def _filter_corrupt_pairs(pairs: list) -> tuple:
    filtered = [(fp, lbl) for fp, lbl in pairs if fp not in corrupt_set]
    removed = len(pairs) - len(filtered)
    return filtered, removed


if LAYOUT == "split":
    train_pairs = _collect_pairs_from_dir(TRAIN_DIR) if TRAIN_DIR else []
    val_pairs = _collect_pairs_from_dir(VALID_DIR) if VALID_DIR else []
    test_pairs = _collect_pairs_from_dir(TEST_DIR) if TEST_DIR else []

    train_pairs, train_removed = _filter_corrupt_pairs(train_pairs)
    val_pairs, val_removed = _filter_corrupt_pairs(val_pairs)
    test_pairs, test_removed = _filter_corrupt_pairs(test_pairs)

    if TEST_DIR is None:
        train_filepaths = [fp for fp, _ in train_pairs]
        train_labels = [lbl for _, lbl in train_pairs]
        new_train_paths, carved_test_paths, new_train_labels, carved_test_labels = train_test_split(
            train_filepaths,
            train_labels,
            test_size=0.15,
            stratify=train_labels,
            random_state=SEED,
        )
        train_pairs = list(zip(new_train_paths, new_train_labels))
        test_pairs = list(zip(carved_test_paths, carved_test_labels))
        print(
            "No test folder found — held out 15% stratified test set from train. "
            "Val folder used as-is for validation."
        )
else:
    all_pairs = []
    for class_name in class_names:
        class_dir = DATA_ROOT_PATH / class_name
        if not class_dir.is_dir():
            continue
        label_idx = class_to_index[class_name]
        for img_path in sorted(class_dir.iterdir(), key=lambda p: (p.name.lower(), p.name)):
            if _is_image_file(img_path):
                all_pairs.append((str(img_path), label_idx))

    filepaths = [fp for fp, _ in all_pairs]
    labels = [lbl for _, lbl in all_pairs]

    train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
        filepaths,
        labels,
        test_size=0.15,
        stratify=labels,
        random_state=SEED,
    )
    val_fraction = 0.15 / 0.85
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        train_val_paths,
        train_val_labels,
        test_size=val_fraction,
        stratify=train_val_labels,
        random_state=SEED,
    )

    train_pairs = list(zip(train_paths, train_labels))
    val_pairs = list(zip(val_paths, val_labels))
    test_pairs = list(zip(test_paths, test_labels))

    train_pairs, train_removed = _filter_corrupt_pairs(train_pairs)
    val_pairs, val_removed = _filter_corrupt_pairs(val_pairs)
    test_pairs, test_removed = _filter_corrupt_pairs(test_pairs)

total_removed = train_removed + val_removed + test_removed

print(
    f"Removed corrupt paths: train={train_removed}, val={val_removed}, "
    f"test={test_removed}, total={total_removed}"
)

random.Random(SEED).shuffle(train_pairs)

train_paths = {p for p, _ in train_pairs}
val_paths = {p for p, _ in val_pairs}
test_paths = {p for p, _ in test_pairs}
assert train_paths.isdisjoint(val_paths), "train/val leakage"
assert train_paths.isdisjoint(test_paths), "train/test leakage"
assert val_paths.isdisjoint(test_paths), "val/test leakage"
print("No split leakage detected.")


def _pairs_contain_corrupt(pairs: list) -> int:
    return sum(1 for fp, _ in pairs if fp in corrupt_set)


corrupt_in_train = _pairs_contain_corrupt(train_pairs)
corrupt_in_val = _pairs_contain_corrupt(val_pairs)
corrupt_in_test = _pairs_contain_corrupt(test_pairs)

print(f"train={len(train_pairs)}, val={len(val_pairs)}, test={len(test_pairs)}")
print(
    f"Corrupt paths remaining (train/val/test): "
    f"{corrupt_in_train}/{corrupt_in_val}/{corrupt_in_test}"
)

split_records = []
for class_name in class_names:
    idx = class_to_index[class_name]
    train_count = sum(1 for _, lbl in train_pairs if lbl == idx)
    val_count = sum(1 for _, lbl in val_pairs if lbl == idx)
    test_count = sum(1 for _, lbl in test_pairs if lbl == idx)
    split_records.append(
        {
            "class": class_name,
            "train": train_count,
            "valid": val_count,
            "test": test_count,
            "total": train_count + val_count + test_count,
        }
    )

split_summary_df = pd.DataFrame(split_records)
print("\nSplit summary:")
print(split_summary_df.to_string(index=False))
split_summary_df.to_csv(f"{RESULTS_DIR}/split_summary.csv", index=False)
print(f"Saved: {RESULTS_DIR}/split_summary.csv")

## 8. Image Loading & Preprocessing Function

In [ ]:
def load_and_preprocess(filepath, label):
    image_bytes = tf.io.read_file(filepath)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    # Keep pixels in [0, 255] float32. Do NOT normalize to [0, 1] here.
    # Per-model preprocessing (Rescaling layer or preprocess_input for MobileNetV2/EfficientNet)
    # will be applied inside each model graph in Features 4-6, keeping custom CNN and transfer models consistent.

    label_one_hot = tf.one_hot(label, NUM_CLASSES)
    return image, label_one_hot


def make_dataset(pairs, training: bool, split_name: str = "unknown"):
    if not pairs:
        print(
            f"WARNING: {split_name} split is empty — returning None "
            f"(downstream code should check `if dataset is not None`)."
        )
        return None

    filepaths = [fp for fp, _ in pairs]
    labels = [lbl for _, lbl in pairs]
    path_ds = tf.data.Dataset.from_tensor_slices(tf.constant(filepaths, dtype=tf.string))
    label_ds = tf.data.Dataset.from_tensor_slices(tf.constant(labels, dtype=tf.int32))
    ds = tf.data.Dataset.zip((path_ds, label_ds))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)

    if training:
        buffer_size = min(len(pairs), 1000)
        ds = ds.shuffle(buffer_size, seed=SEED)
        # data_augmentation is defined in the next cell; resolved at call time when training=True.
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

## 9. Data Augmentation

In [ ]:
# Active only in training pipelines (make_dataset(..., training=True)); bypassed for val/test.
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.15, seed=SEED),
        tf.keras.layers.RandomZoom(0.15, seed=SEED),
        tf.keras.layers.RandomTranslation(0.1, 0.1, seed=SEED),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255), seed=SEED),
        tf.keras.layers.RandomContrast(0.15, seed=SEED),
        # Clip back to [0, 255] so brightness/contrast cannot push pixels out of range
        # before per-model preprocess_input is applied in Features 4-6.
        tf.keras.layers.Lambda(lambda x: tf.clip_by_value(x, 0.0, 255.0)),
    ],
    name="data_augmentation",
)

## 10. Class Weights (Imbalance Handling)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = np.array([lbl for _, lbl in train_pairs])
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels,
)
class_weights_dict = {int(idx): float(weight) for idx, weight in enumerate(class_weight_values)}

print("Class weights (balanced, from train split) — by class name:")
for idx in range(NUM_CLASSES):
    print(f"  {index_to_class[idx]}: {class_weights_dict[idx]:.4f}")

class_weights_path = f"{RESULTS_DIR}/class_weights.json"
with open(class_weights_path, "w", encoding="utf-8") as f:
    json.dump({str(k): float(v) for k, v in class_weights_dict.items()}, f, indent=2)
print(f"Saved: {class_weights_path}")

## 11. Build Datasets & Sanity Check

In [ ]:
train_ds = make_dataset(train_pairs, training=True, split_name="train")

if val_pairs:
    val_ds = make_dataset(val_pairs, training=False, split_name="val")
else:
    val_ds = None
    print("val split absent — val_ds=None")

if test_pairs:
    test_ds = make_dataset(test_pairs, training=False, split_name="test")
else:
    test_ds = None
    print("test split absent — test_ds=None")

if train_ds is not None:
    print("\n--- Augmented train batch sanity check ---")
    for images_batch, labels_batch in train_ds.take(1):
        print(f"Image batch shape: {images_batch.shape}")
        print(f"Image dtype: {images_batch.dtype}")
        print(
            f"Pixel min/max: {tf.reduce_min(images_batch):.2f} / {tf.reduce_max(images_batch):.2f}"
        )
        print(f"Label batch shape: {labels_batch.shape}")
        row_sums = tf.reduce_sum(labels_batch, axis=1)
        print(
            "One-hot row sums (should all be 1): "
            f"min={tf.reduce_min(row_sums):.4f}, max={tf.reduce_max(row_sums):.4f}"
        )

        n_show = min(9, images_batch.shape[0])
        ncols = 3
        nrows = int(np.ceil(n_show / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
        axes = np.atleast_1d(axes).flatten()

        for i in range(n_show):
            img = images_batch[i].numpy()
            img_display = np.clip(img, 0, 255).astype(np.uint8)
            label_idx = int(tf.argmax(labels_batch[i]).numpy())
            axes[i].imshow(img_display)
            axes[i].set_title(index_to_class[label_idx], fontsize=8)
            axes[i].axis("off")

        for i in range(n_show, len(axes)):
            axes[i].axis("off")

        plt.tight_layout()
        aug_path = f"{RESULTS_DIR}/augmented_samples.png"
        plt.savefig(aug_path, dpi=150)
        plt.show()
        print(f"Saved: {aug_path}")

print("\n--- Non-augmented preprocessing sanity check ---")
if val_ds is not None:
    check_ds = val_ds
    check_name = "val_ds"
else:
    check_ds = make_dataset(train_pairs, training=False, split_name="train (no-aug check)")
    check_name = "train_ds (training=False)"

if check_ds is not None:
    for images_batch, labels_batch in check_ds.take(1):
        print(f"Source: {check_name}")
        print(f"Image batch shape: {images_batch.shape}")
        print(f"Image dtype: {images_batch.dtype}")
        print(
            f"Pixel min/max: {tf.reduce_min(images_batch):.2f} / {tf.reduce_max(images_batch):.2f}"
        )
        print(f"Label batch shape: {labels_batch.shape}")
        row_sums = tf.reduce_sum(labels_batch, axis=1)
        print(
            "One-hot row sums (should all be 1): "
            f"min={tf.reduce_min(row_sums):.4f}, max={tf.reduce_max(row_sums):.4f}"
        )
else:
    print("No non-augmented dataset available for preprocessing check.")

## 12. Model 1 — Custom CNN (Baseline)

Baseline conv net built from scratch. Input pixels arrive in `[0, 255]` float32 from the data pipeline, so a `Rescaling(1./255)` layer normalizes inside the model graph.

In [ ]:
def reset_seeds() -> None:
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)


if "EPOCHS" not in globals():
    EPOCHS = 30
if "EARLY_STOP_PATIENCE" not in globals():
    EARLY_STOP_PATIENCE = 5

In [ ]:
from tensorflow.keras import layers, models, optimizers


def build_custom_cnn(input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape, name="input_image")
    x = layers.Rescaling(1.0 / 255.0, name="rescaling")(inputs)

    conv_filters = [32, 64, 128, 256]
    for i, filters in enumerate(conv_filters, start=1):
        x = layers.Conv2D(
            filters,
            kernel_size=3,
            padding="same",
            activation=None,
            name=f"conv{i}_a",
        )(x)
        x = layers.BatchNormalization(name=f"bn{i}_a")(x)
        x = layers.Activation("relu", name=f"relu{i}_a")(x)
        x = layers.Conv2D(
            filters,
            kernel_size=3,
            padding="same",
            activation=None,
            name=f"conv{i}_b",
        )(x)
        x = layers.BatchNormalization(name=f"bn{i}_b")(x)
        x = layers.Activation("relu", name=f"relu{i}_b")(x)
        x = layers.MaxPooling2D(pool_size=2, name=f"pool{i}")(x)
        x = layers.Dropout(0.25, name=f"dropout_conv{i}")(x)

    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.BatchNormalization(name="bn_dense")(x)
    x = layers.Dropout(0.5, name="dropout_dense")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="custom_cnn")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


reset_seeds()
custom_cnn_model = build_custom_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
custom_cnn_model.summary()

total_params = custom_cnn_model.count_params()
trainable_params = sum(
    tf.keras.backend.count_params(w) for w in custom_cnn_model.trainable_weights
)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


def make_callbacks(model_name: str):
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True,
        ),
        ModelCheckpoint(
            filepath=f"{RESULTS_DIR}/{model_name}_best.keras",
            monitor="val_accuracy",
            save_best_only=True,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

### Train Custom CNN

In [ ]:
import time

reset_seeds()
train_start = time.time()

assert val_ds is not None, (
    "val_ds is None — callbacks monitor val_loss/val_accuracy and require a validation set."
)

history_custom = custom_cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=make_callbacks("custom_cnn"),
    class_weight=class_weights_dict,
)

training_time_custom = time.time() - train_start
print(
    f"Training time: {training_time_custom:.1f} seconds "
    f"({training_time_custom / 60:.2f} minutes)"
)

best_epoch_idx = int(np.argmax(history_custom.history["val_accuracy"]))
best_epoch = best_epoch_idx + 1
best_val_acc_custom = history_custom.history["val_accuracy"][best_epoch_idx]
best_val_loss_at_best_acc = history_custom.history["val_loss"][best_epoch_idx]
final_train_acc = history_custom.history["accuracy"][-1]
print(
    f"Custom CNN — best val_accuracy={best_val_acc_custom:.4f} at epoch {best_epoch} | "
    f"val_loss={best_val_loss_at_best_acc:.4f} | final train_acc={final_train_acc:.4f}"
)

history_custom_path = f"{RESULTS_DIR}/history_custom_cnn.json"
with open(history_custom_path, "w", encoding="utf-8") as f:
    json.dump(history_custom.history, f, indent=2)
print(f"Saved: {history_custom_path}")

### Training Curve — Custom CNN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_custom.history["accuracy"], label="Train")
axes[0].plot(history_custom.history["val_accuracy"], label="Validation")
axes[0].set_title("Custom CNN — Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_custom.history["loss"], label="Train")
axes[1].plot(history_custom.history["val_loss"], label="Validation")
axes[1].set_title("Custom CNN — Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = f"{RESULTS_DIR}/training_curve_custom_cnn.png"
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved: {curve_path}")


## 13. Model 2 — MobileNetV2 (Transfer Learning)

Pretrained ImageNet base, frozen first for feature extraction, then fine-tuned on the top layers. Input pixels arrive in `[0, 255]` float32 and are scaled to `[-1, 1]` via `mobilenet_v2.preprocess_input` inside the model graph.

In [ ]:
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


def build_mobilenetv2(input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape, name="input_image")
    x = preprocess_input(inputs)
    base = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
    )
    base.trainable = False
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(base.output)
    x = layers.Dropout(0.3, name="dropout_head")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="mobilenetv2")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base


reset_seeds()
mnv2_model, mnv2_base = build_mobilenetv2(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
mnv2_model.summary()

trainable_params = sum(
    tf.keras.backend.count_params(w) for w in mnv2_model.trainable_weights
)
print(f"Trainable params (frozen base): {trainable_params:,}")

### Phase 1 — Feature Extraction (frozen base)

In [ ]:
reset_seeds()
assert val_ds is not None, (
    "val_ds is None — callbacks monitor val_loss/val_accuracy and require a validation set."
)

PHASE1_EPOCHS = 15
phase1_start = time.time()

history_mnv2_p1 = mnv2_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=make_callbacks("mobilenetv2_p1"),
    class_weight=class_weights_dict,
)

phase1_time_mnv2 = time.time() - phase1_start
print(f"Phase 1 time: {phase1_time_mnv2:.1f}s ({phase1_time_mnv2 / 60:.2f} min)")

### Phase 2 — Fine-Tuning (unfreeze top layers)

In [ ]:
mnv2_base.trainable = True
for layer in mnv2_base.layers[:-30]:
    layer.trainable = False

for layer in mnv2_base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_layer_count = sum(1 for layer in mnv2_base.layers if layer.trainable)
for layer in mnv2_model.layers:
    if layer is mnv2_base:
        continue
    if layer.trainable:
        trainable_layer_count += 1
print(f"Trainable layers after BN freeze adjustment: {trainable_layer_count}")

mnv2_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

reset_seeds()

FINE_TUNE_EPOCHS = 10
phase2_start = time.time()

history_mnv2_p2 = mnv2_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=make_callbacks("mobilenetv2_finetune"),
    class_weight=class_weights_dict,
)

phase2_time_mnv2 = time.time() - phase2_start
training_time_mnv2 = phase1_time_mnv2 + phase2_time_mnv2
print(f"Phase 2 time: {phase2_time_mnv2:.1f}s ({phase2_time_mnv2 / 60:.2f} min)")
print(
    f"Total MobileNetV2 training time: {training_time_mnv2:.1f}s "
    f"({training_time_mnv2 / 60:.2f} min)"
)

In [ ]:
import shutil

history_mnv2_combined = {
    "accuracy": history_mnv2_p1.history["accuracy"] + history_mnv2_p2.history["accuracy"],
    "val_accuracy": history_mnv2_p1.history["val_accuracy"] + history_mnv2_p2.history["val_accuracy"],
    "loss": history_mnv2_p1.history["loss"] + history_mnv2_p2.history["loss"],
    "val_loss": history_mnv2_p1.history["val_loss"] + history_mnv2_p2.history["val_loss"],
}

fine_tune_boundary = len(history_mnv2_p1.history["accuracy"])

p1_best_idx = int(np.argmax(history_mnv2_p1.history["val_accuracy"]))
p2_best_idx = int(np.argmax(history_mnv2_p2.history["val_accuracy"]))
p1_best_val_acc = history_mnv2_p1.history["val_accuracy"][p1_best_idx]
p2_best_val_acc = history_mnv2_p2.history["val_accuracy"][p2_best_idx]

p1_ckpt = f"{RESULTS_DIR}/mobilenetv2_p1_best.keras"
p2_ckpt = f"{RESULTS_DIR}/mobilenetv2_finetune_best.keras"
mnv2_best_path = f"{RESULTS_DIR}/mobilenetv2_best.keras"

if p2_best_val_acc >= p1_best_val_acc:
    best_phase_num = 2
    best_val_acc_mnv2 = p2_best_val_acc
    best_val_loss_mnv2 = history_mnv2_p2.history["val_loss"][p2_best_idx]
    best_epoch = p2_best_idx + 1
    winning_ckpt = p2_ckpt
else:
    best_phase_num = 1
    best_val_acc_mnv2 = p1_best_val_acc
    best_val_loss_mnv2 = history_mnv2_p1.history["val_loss"][p1_best_idx]
    best_epoch = p1_best_idx + 1
    winning_ckpt = p1_ckpt

shutil.copy(winning_ckpt, mnv2_best_path)
print(
    f"MobileNetV2 best = phase {best_phase_num} (val_acc={best_val_acc_mnv2:.4f}) "
    f"-> saved as mobilenetv2_best.keras"
)

history_mnv2_export = {
    **history_mnv2_combined,
    "fine_tune_boundary": fine_tune_boundary,
    "best": {
        "best_val_accuracy": float(best_val_acc_mnv2),
        "best_val_loss": float(best_val_loss_mnv2),
        "best_phase": best_phase_num,
        "best_epoch": best_epoch,
        "total_training_time_seconds": float(training_time_mnv2),
    },
}

history_mnv2_path = f"{RESULTS_DIR}/history_mobilenetv2.json"
with open(history_mnv2_path, "w", encoding="utf-8") as f:
    json.dump(history_mnv2_export, f, indent=2)
print(f"Saved: {history_mnv2_path}")

print(
    f"MobileNetV2 — best val_accuracy={best_val_acc_mnv2:.4f} at (phase {best_phase_num}, epoch {best_epoch}) | "
    f"val_loss={best_val_loss_mnv2:.4f}"
)

### Training Curve — MobileNetV2

In [ ]:
epochs = range(1, len(history_mnv2_combined["accuracy"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history_mnv2_combined["accuracy"], label="Train")
axes[0].plot(epochs, history_mnv2_combined["val_accuracy"], label="Validation")
axes[0].axvline(
    fine_tune_boundary + 0.5,
    color="gray",
    linestyle="--",
    label="fine-tune start",
)
axes[0].set_title("MobileNetV2 — Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history_mnv2_combined["loss"], label="Train")
axes[1].plot(epochs, history_mnv2_combined["val_loss"], label="Validation")
axes[1].axvline(
    fine_tune_boundary + 0.5,
    color="gray",
    linestyle="--",
    label="fine-tune start",
)
axes[1].set_title("MobileNetV2 — Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = f"{RESULTS_DIR}/training_curve_mobilenetv2.png"
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved: {curve_path}")

## 14. Model 3 — EfficientNetB0 (Transfer Learning)

Pretrained ImageNet base, frozen first for feature extraction, then fine-tuned on the top layers. Input pixels arrive in `[0, 255]` float32; EfficientNet's `preprocess_input` expects raw `[0, 255]` and normalizes internally (different from MobileNetV2's `[-1, 1]` scaling).

In [ ]:
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess


def build_efficientnetb0(input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape, name="input_image")
    # EfficientNet expects raw [0, 255] pixels and applies its own normalization
    # internally, unlike MobileNetV2 which needs inputs scaled to [-1, 1].
    x = effnet_preprocess(inputs)
    base = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
    )
    base.trainable = False
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(base.output)
    x = layers.Dropout(0.3, name="dropout_head")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="efficientnetb0")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base


reset_seeds()
eff_model, eff_base = build_efficientnetb0(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
eff_model.summary()

trainable_params = sum(
    tf.keras.backend.count_params(w) for w in eff_model.trainable_weights
)
print(f"Trainable params (frozen base): {trainable_params:,}")

### Phase 1 — Feature Extraction (frozen base)

In [ ]:
reset_seeds()
assert val_ds is not None, (
    "val_ds is None — callbacks monitor val_loss/val_accuracy and require a validation set."
)

PHASE1_EPOCHS = 15
phase1_start = time.time()

history_eff_p1 = eff_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=make_callbacks("efficientnetb0_p1"),
    class_weight=class_weights_dict,
)

phase1_time_eff = time.time() - phase1_start
print(f"Phase 1 time: {phase1_time_eff:.1f}s ({phase1_time_eff / 60:.2f} min)")

### Phase 2 — Fine-Tuning (unfreeze top layers, keep BatchNorm frozen)

In [ ]:
eff_base.trainable = True
for layer in eff_base.layers[:-30]:
    layer.trainable = False

# EfficientNet has many BatchNormalization layers; unfreezing them destabilizes
# fine-tuning, so keep every BN layer frozen even within the top unfrozen block.
for layer in eff_base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_layer_count = sum(1 for layer in eff_base.layers if layer.trainable)
for layer in eff_model.layers:
    if layer is eff_base:
        continue
    if layer.trainable:
        trainable_layer_count += 1
print(f"Trainable layers after BN freeze adjustment: {trainable_layer_count}")

eff_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

reset_seeds()

FINE_TUNE_EPOCHS = 10
phase2_start = time.time()

history_eff_p2 = eff_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=make_callbacks("efficientnetb0_finetune"),
    class_weight=class_weights_dict,
)

phase2_time_eff = time.time() - phase2_start
training_time_eff = phase1_time_eff + phase2_time_eff
print(f"Phase 2 time: {phase2_time_eff:.1f}s ({phase2_time_eff / 60:.2f} min)")
print(
    f"Total EfficientNetB0 training time: {training_time_eff:.1f}s "
    f"({training_time_eff / 60:.2f} min)"
)

In [ ]:
import shutil

history_eff_combined = {
    "accuracy": history_eff_p1.history["accuracy"] + history_eff_p2.history["accuracy"],
    "val_accuracy": history_eff_p1.history["val_accuracy"] + history_eff_p2.history["val_accuracy"],
    "loss": history_eff_p1.history["loss"] + history_eff_p2.history["loss"],
    "val_loss": history_eff_p1.history["val_loss"] + history_eff_p2.history["val_loss"],
}

fine_tune_boundary = len(history_eff_p1.history["accuracy"])

p1_best_idx = int(np.argmax(history_eff_p1.history["val_accuracy"]))
p2_best_idx = int(np.argmax(history_eff_p2.history["val_accuracy"]))
p1_best_val_acc = history_eff_p1.history["val_accuracy"][p1_best_idx]
p2_best_val_acc = history_eff_p2.history["val_accuracy"][p2_best_idx]

p1_ckpt = f"{RESULTS_DIR}/efficientnetb0_p1_best.keras"
p2_ckpt = f"{RESULTS_DIR}/efficientnetb0_finetune_best.keras"
eff_best_path = f"{RESULTS_DIR}/efficientnetb0_best.keras"

if p2_best_val_acc >= p1_best_val_acc:
    best_phase_num = 2
    best_val_acc_eff = p2_best_val_acc
    best_val_loss_eff = history_eff_p2.history["val_loss"][p2_best_idx]
    best_epoch = p2_best_idx + 1
    winning_ckpt = p2_ckpt
else:
    best_phase_num = 1
    best_val_acc_eff = p1_best_val_acc
    best_val_loss_eff = history_eff_p1.history["val_loss"][p1_best_idx]
    best_epoch = p1_best_idx + 1
    winning_ckpt = p1_ckpt

shutil.copy(winning_ckpt, eff_best_path)
print(
    f"EfficientNetB0 best = phase {best_phase_num} (val_acc={best_val_acc_eff:.4f}) "
    f"-> saved as efficientnetb0_best.keras"
)

history_eff_export = {
    **history_eff_combined,
    "fine_tune_boundary": fine_tune_boundary,
    "best": {
        "best_val_accuracy": float(best_val_acc_eff),
        "best_val_loss": float(best_val_loss_eff),
        "best_phase": best_phase_num,
        "best_epoch": best_epoch,
        "total_training_time_seconds": float(training_time_eff),
    },
}

history_eff_path = f"{RESULTS_DIR}/history_efficientnetb0.json"
with open(history_eff_path, "w", encoding="utf-8") as f:
    json.dump(history_eff_export, f, indent=2)
print(f"Saved: {history_eff_path}")

print(
    f"EfficientNetB0 — best val_accuracy={best_val_acc_eff:.4f} at (phase {best_phase_num}, epoch {best_epoch}) | "
    f"val_loss={best_val_loss_eff:.4f}"
)

### Training Curve — EfficientNetB0

In [ ]:
epochs = range(1, len(history_eff_combined["accuracy"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history_eff_combined["accuracy"], label="Train")
axes[0].plot(epochs, history_eff_combined["val_accuracy"], label="Validation")
axes[0].axvline(
    fine_tune_boundary + 0.5,
    color="gray",
    linestyle="--",
    label="fine-tune start",
)
axes[0].set_title("EfficientNetB0 — Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history_eff_combined["loss"], label="Train")
axes[1].plot(epochs, history_eff_combined["val_loss"], label="Validation")
axes[1].axvline(
    fine_tune_boundary + 0.5,
    color="gray",
    linestyle="--",
    label="fine-tune start",
)
axes[1].set_title("EfficientNetB0 — Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = f"{RESULTS_DIR}/training_curve_efficientnetb0.png"
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved: {curve_path}")

## 15. Evaluation & Model Comparison

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

if test_ds is not None:
    EVAL_DS = test_ds
    EVAL_DS_NAME = "test"
else:
    EVAL_DS = val_ds
    EVAL_DS_NAME = "val (test absent)"
    print("NOTE: test_ds is None — falling back to val_ds for evaluation.")

assert EVAL_DS is not None, "No evaluation dataset available (both test_ds and val_ds are None)."
print(f"Evaluation split: {EVAL_DS_NAME}")

# Materialize y_true ONCE from the (unshuffled) eval pipeline so it aligns with predictions.
y_true_all = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in EVAL_DS])
n_eval_images = int(y_true_all.shape[0])
print(f"Evaluation images: {n_eval_images}")


def import_and_eval(model_path, model_name):
    model = tf.keras.models.load_model(model_path, compile=False)
    start = time.time()
    probs = model.predict(EVAL_DS, verbose=0)
    total_ms = (time.time() - start) * 1000.0
    y_pred = np.argmax(probs, axis=1)
    per_image_ms = total_ms / max(n_eval_images, 1)
    return {
        "name": model_name,
        "y_pred": y_pred,
        "probs": probs,
        "accuracy": accuracy_score(y_true_all, y_pred),
        "macro_f1": f1_score(y_true_all, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true_all, y_pred, average="weighted", zero_division=0),
        "precision_macro": precision_score(y_true_all, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true_all, y_pred, average="macro", zero_division=0),
        "infer_ms_per_image": per_image_ms,
        "params": int(model.count_params()),
        "size_mb": os.path.getsize(model_path) / 1e6,
    }

### Per-Model Evaluation

In [ ]:
MODEL_CHECKPOINTS = [
    ("Custom CNN", f"{RESULTS_DIR}/custom_cnn_best.keras"),
    ("MobileNetV2", f"{RESULTS_DIR}/mobilenetv2_best.keras"),
    ("EfficientNetB0", f"{RESULTS_DIR}/efficientnetb0_best.keras"),
]

eval_results = []
for model_name, model_path in MODEL_CHECKPOINTS:
    if not os.path.exists(model_path):
        print(f"WARNING: checkpoint missing for {model_name}: {model_path} — skipping.")
        continue
    try:
        res = import_and_eval(model_path, model_name)
        eval_results.append(res)
        print(
            f"{model_name}: acc={res['accuracy']:.4f} | macroF1={res['macro_f1']:.4f} | "
            f"weightedF1={res['weighted_f1']:.4f} | {res['infer_ms_per_image']:.2f} ms/img | "
            f"{res['params']:,} params | {res['size_mb']:.1f} MB"
        )
    except Exception as e:
        print(f"WARNING: failed to evaluate {model_name} ({model_path}): {e}")

assert eval_results, "No models could be evaluated — check that best checkpoints exist in RESULTS_DIR."

### Model Comparison Table

In [ ]:
train_time_lookup = {}
for _name, _var in [
    ("Custom CNN", "training_time_custom"),
    ("MobileNetV2", "training_time_mnv2"),
    ("EfficientNetB0", "training_time_eff"),
]:
    train_time_lookup[_name] = float(globals()[_var]) if _var in globals() else np.nan

comparison_df = pd.DataFrame(
    [
        {
            "Model": r["name"],
            "Test Accuracy": r["accuracy"],
            "Macro F1": r["macro_f1"],
            "Weighted F1": r["weighted_f1"],
            "Precision (macro)": r["precision_macro"],
            "Recall (macro)": r["recall_macro"],
            "Inference ms/img": r["infer_ms_per_image"],
            "Params": r["params"],
            "Size (MB)": r["size_mb"],
            "Train Time (s)": train_time_lookup.get(r["name"], np.nan),
        }
        for r in eval_results
    ]
)
comparison_df = comparison_df.sort_values("Macro F1", ascending=False).reset_index(drop=True)
print(f"Model comparison on {EVAL_DS_NAME} split:")
print(comparison_df.to_string(index=False))

comparison_csv = f"{RESULTS_DIR}/model_comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)
print(f"Saved: {comparison_csv}")

### Best Model Selection

In [ ]:
# Tie-break: highest Macro F1, then Test Accuracy, then smaller Size (MB).
best_row = comparison_df.sort_values(
    by=["Macro F1", "Test Accuracy", "Size (MB)"],
    ascending=[False, False, True],
).iloc[0]
BEST_MODEL_NAME = best_row["Model"]

_name_to_path = dict(MODEL_CHECKPOINTS)
BEST_MODEL_PATH = _name_to_path[BEST_MODEL_NAME]

best_result = next(r for r in eval_results if r["name"] == BEST_MODEL_NAME)
best_y_pred = best_result["y_pred"]
best_probs = best_result["probs"]

print(
    f"Selected best model: {BEST_MODEL_NAME} "
    f"(Macro F1={best_row['Macro F1']:.4f}, Acc={best_row['Test Accuracy']:.4f})"
)
print(f"BEST_MODEL_PATH: {BEST_MODEL_PATH}")

### Classification Report — Best Model

In [ ]:
report_dict = classification_report(
    y_true_all,
    best_y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
print(f"Classification report — {BEST_MODEL_NAME} on {EVAL_DS_NAME} split:")
print(report_df.to_string())

report_csv = f"{RESULTS_DIR}/classification_report.csv"
report_df.to_csv(report_csv)
print(f"Saved: {report_csv}")

print(f"Macro F1:    {report_dict['macro avg']['f1-score']:.4f}")
print(f"Weighted F1: {report_dict['weighted avg']['f1-score']:.4f}")

### Confusion Matrix — Best Model

In [ ]:
cm = confusion_matrix(y_true_all, best_y_pred, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, axes = plt.subplots(1, 2, figsize=(2 * max(8, NUM_CLASSES), max(7, NUM_CLASSES)))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[0],
)
axes[0].set_title(f"Confusion Matrix (counts) — {BEST_MODEL_NAME} [{EVAL_DS_NAME}]")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[1],
)
axes[1].set_title(f"Row-normalized (per true class) — {BEST_MODEL_NAME}")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
cm_path = f"{RESULTS_DIR}/confusion_matrix.png"
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved: {cm_path}")

# RQ4: top-3 most-confused class pairs (highest off-diagonal counts).
off = [
    (i, j, int(cm[i, j]))
    for i in range(NUM_CLASSES)
    for j in range(NUM_CLASSES)
    if i != j
]
off.sort(key=lambda t: t[2], reverse=True)
print("\nTop 3 most-confused class pairs (true -> predicted, count):")
for i, j, c in off[:3]:
    print(f"  {index_to_class[i]} -> {index_to_class[j]}: {c}")

## 16. Grad-CAM Explainability

Heatmaps are computed on the best model loaded from `BEST_MODEL_PATH`. Preprocessing is inside the model graph, so raw `[0, 255]` images are fed directly.

In [ ]:
gradcam_model = tf.keras.models.load_model(BEST_MODEL_PATH, compile=False)
print(f"Loaded best model for Grad-CAM: {BEST_MODEL_NAME} ({BEST_MODEL_PATH})")

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mnv2_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess


def _get_nested_preprocess_fn(model_name):
    if "MobileNet" in model_name:
        print("Nested two-stage preprocessing: mobilenet_v2.preprocess_input ([0,255] -> [-1,1])")
        return mnv2_preprocess
    if "EfficientNet" in model_name:
        print("Nested two-stage preprocessing: efficientnet.preprocess_input ([0,255] internal norm)")
        return effnet_preprocess
    raise ValueError(
        f"Nested two-stage path requires MobileNetV2 or EfficientNetB0; got {model_name}"
    )


def _print_all_layer_names(model, prefix=""):
    for layer in model.layers:
        print(f"  {prefix}{layer.name} -> {getattr(layer, 'output', None)}")
        if isinstance(layer, tf.keras.Model):
            _print_all_layer_names(layer, prefix=f"{prefix}{layer.name}/")


def find_last_conv_layer(model):
    def _search(m, host):
        for layer in reversed(m.layers):
            if isinstance(layer, tf.keras.Model):
                result = _search(layer, layer)
                if result is not None:
                    return result
            try:
                shape = layer.output.shape
            except AttributeError:
                continue
            if len(shape) == 4:
                return layer, host
        return None

    return _search(model, model)


def _smoke_test_nested_two_stage(base_feat_model, head_layers, preprocess_fn, img_size):
    dummy = tf.zeros((1, *img_size, 3), dtype=tf.float32)
    with tf.GradientTape() as tape:
        x = preprocess_fn(dummy)
        conv_out, base_out = base_feat_model(x, training=False)
        tape.watch(conv_out)
        h = base_out
        for layer in head_layers:
            h = layer(h, training=False)
        class_channel = h[:, 0]
    grads = tape.gradient(class_channel, conv_out)
    if conv_out is None or grads is None:
        raise RuntimeError("nested two-stage smoke test failed: missing conv_out or grads")
    print(
        f"nested two-stage fallback: OK (conv_out {conv_out.shape}, grads {grads.shape})"
    )


def build_gradcam_setup(model, conv_layer, host_model, model_name=BEST_MODEL_NAME):
    head_layers = []
    if host_model is not model:
        base_idx = next(
            i
            for i, layer in enumerate(model.layers)
            if layer is host_model or layer.name == host_model.name
        )
        head_layers = list(model.layers[base_idx + 1 :])

    # CASE A — conv layer in the top-level model (Custom CNN).
    if host_model is model:
        try:
            grad_model = tf.keras.models.Model(
                model.inputs, [conv_layer.output, model.output]
            )
            print(
                f"Grad-CAM construction path: top-level | conv layer: {conv_layer.name} "
                f"(output {conv_layer.output.shape})"
            )
            return {"path": "top-level", "grad_model": grad_model}
        except Exception as e:
            print(f"top-level grad model build failed: {e}")
            _print_all_layer_names(model)
            raise

    base_name = host_model.name
    base_model = model.get_layer(base_name)
    inner_conv = base_model.get_layer(conv_layer.name)

    # CASE B — try functional nested reference (reconnects inner conv to outer input graph).
    try:
        nested_conv_out = model.get_layer(base_name).get_layer(conv_layer.name).output
        grad_model = tf.keras.models.Model(
            inputs=model.inputs,
            outputs=[nested_conv_out, model.output],
        )
        _ = grad_model(tf.zeros((1, *IMG_SIZE, 3), dtype=tf.float32), training=False)
        print(
            f"Grad-CAM construction path: nested-functional | conv layer: {inner_conv.name} "
            f"(output {nested_conv_out.shape})"
        )
        return {"path": "nested-functional", "grad_model": grad_model}
    except Exception as e:
        print(f"nested-functional grad model build failed: {e} — falling back to nested two-stage")

    # CASE B fallback — true two-stage on the base's OWN input/output graph.
    try:
        preprocess_fn = _get_nested_preprocess_fn(model_name)
        base_feat_model = tf.keras.models.Model(
            inputs=base_model.inputs,
            outputs=[inner_conv.output, base_model.output],
        )
        _smoke_test_nested_two_stage(base_feat_model, head_layers, preprocess_fn, IMG_SIZE)
        print(
            f"Grad-CAM construction path: nested two-stage | conv layer: {inner_conv.name} "
            f"| head layers: {[layer.name for layer in head_layers]}"
        )
        return {
            "path": "nested two-stage",
            "base_feat_model": base_feat_model,
            "head_layers": head_layers,
            "preprocess_fn": preprocess_fn,
        }
    except Exception as e:
        print(f"nested two-stage grad model build failed: {e}")
        print("Available layers (including nested base):")
        _print_all_layer_names(model)
        raise


result = find_last_conv_layer(gradcam_model)
if result is None:
    print("Could not auto-detect a 4D conv layer. Available layers:")
    _print_all_layer_names(gradcam_model)
    raise RuntimeError("No 4D conv feature-map layer found for Grad-CAM.")

last_conv_layer, host_model = result
try:
    gradcam_setup = build_gradcam_setup(gradcam_model, last_conv_layer, host_model)
except Exception as e:
    print(f"Grad-CAM setup failed: {e}")
    raise

In [ ]:
def make_gradcam_heatmap(img_array, setup, pred_index):
    # img_array: raw [0,255] batch of shape (1, H, W, 3); preprocessing is in-graph for top-level paths.
    img_array = tf.cast(img_array, tf.float32)

    if setup["path"] in ("top-level", "nested-functional"):
        grad_model = setup["grad_model"]
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(img_array, training=False)
            class_channel = preds[:, pred_index]
        grads = tape.gradient(class_channel, conv_out)
    else:
        base_feat_model = setup["base_feat_model"]
        head_layers = setup["head_layers"]
        preprocess_fn = setup["preprocess_fn"]
        with tf.GradientTape() as tape:
            x = preprocess_fn(img_array)
            conv_out, base_out = base_feat_model(x, training=False)
            tape.watch(conv_out)
            h = base_out
            for layer in head_layers:
                h = layer(h, training=False)
            preds = h
            class_channel = preds[:, pred_index]
        grads = tape.gradient(class_channel, conv_out)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val
    return heatmap.numpy()

In [ ]:
from matplotlib import cm as mpl_cm


def overlay_heatmap(orig_img_uint8, heatmap, alpha=0.4):
    h, w = orig_img_uint8.shape[:2]
    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis], (h, w), method="bilinear"
    ).numpy()[..., 0]
    colored = mpl_cm.get_cmap("jet")(heatmap_resized)[..., :3]  # RGB in [0, 1]
    colored = (colored * 255).astype(np.uint8)
    blended = (alpha * colored + (1 - alpha) * orig_img_uint8).astype(np.uint8)
    return blended

### Grad-CAM on Correct & Incorrect Predictions

In [ ]:
# Select up to 6 correct (spread across classes) and up to 3 incorrect samples,
# using the aligned y_true_all / best_y_pred from Section 15.
correct_mask = best_y_pred == y_true_all
correct_idx = np.where(correct_mask)[0]
incorrect_idx = np.where(~correct_mask)[0]

chosen_correct = []
seen_classes = set()
for i in correct_idx:
    cls = int(y_true_all[i])
    if cls not in seen_classes:
        chosen_correct.append(int(i))
        seen_classes.add(cls)
    if len(chosen_correct) >= 6:
        break
for i in correct_idx:  # top up if not enough distinct classes
    if len(chosen_correct) >= 6:
        break
    if int(i) not in chosen_correct:
        chosen_correct.append(int(i))

chosen_incorrect = [int(i) for i in incorrect_idx[:3]]
if not chosen_incorrect:
    print("NOTE: no incorrect predictions on this split — showing correct samples only.")

chosen = chosen_correct + chosen_incorrect
chosen_set = set(chosen)

# Grab the raw [0,255] images for the chosen global indices by re-iterating EVAL_DS in order.
images_by_idx = {}
cursor = 0
for imgs, _ in EVAL_DS:
    bs = imgs.shape[0]
    for b in range(bs):
        g = cursor + b
        if g in chosen_set:
            images_by_idx[g] = imgs[b].numpy()
    cursor += bs
    if len(images_by_idx) == len(chosen_set):
        break

n = len(chosen)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).flatten()

for ax_i, g in enumerate(chosen):
    raw = images_by_idx[g]
    img_uint8 = np.clip(raw, 0, 255).astype(np.uint8)
    pred_c = int(best_y_pred[g])
    true_c = int(y_true_all[g])
    heatmap = make_gradcam_heatmap(raw[np.newaxis, ...], gradcam_setup, pred_c)
    blended = overlay_heatmap(img_uint8, heatmap, alpha=0.4)
    is_correct = pred_c == true_c
    status = "correct" if is_correct else "incorrect"
    axes[ax_i].imshow(blended)
    axes[ax_i].set_title(
        f"[{status}] true: {index_to_class[true_c]} | pred: {index_to_class[pred_c]}",
        color="green" if is_correct else "red",
        fontsize=9,
    )
    axes[ax_i].axis("off")

for ax_i in range(n, len(axes)):
    axes[ax_i].axis("off")

fig.suptitle(f"Grad-CAM — {BEST_MODEL_NAME} [{EVAL_DS_NAME}]", fontsize=12)
plt.tight_layout()
gradcam_path = f"{RESULTS_DIR}/gradcam_examples.png"
plt.savefig(gradcam_path, dpi=150)
plt.show()
print(f"Saved: {gradcam_path}")
print(
    "RQ5 interpretation prompt: inspect whether the warm (red/yellow) Grad-CAM regions "
    "concentrate on leaf lesion/spot areas versus background/healthy tissue. "
    "Base the report conclusion on the actual saved image — no conclusion is fabricated here."
)

## Explainability — SHAP (Best Model)

SHAP is run on a small sample due to computational cost. Grad-CAM is the primary XAI method; SHAP provides complementary region-attribution evidence on the best model loaded from `BEST_MODEL_PATH`. Preprocessing is in-graph — feed raw `[0, 255]` images.

In [ ]:
SHAP_DONE = False
SHAP_MAX_BACKGROUND = 25
SHAP_MAX_EXPLAIN = 4
SHAP_TIME_LIMIT_SEC = 240  # timeout-style guard (~4 min)

import sys
import signal
import subprocess

import matplotlib.pyplot as plt


def _shap_skip(reason):
    global SHAP_DONE
    SHAP_DONE = False
    print(f"SHAP skipped: {reason}")


try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "shap"],
        check=False,
        capture_output=True,
        timeout=120,
    )
except Exception as exc:
    _shap_skip(f"pip install failed ({exc})")
else:
    try:
        import shap
    except Exception as exc:
        _shap_skip(f"import failed ({exc})")
    else:
        def _run_shap():
            global SHAP_DONE

            shap_model = tf.keras.models.load_model(BEST_MODEL_PATH, compile=False)
            print(
                f"Loaded best model for SHAP: {BEST_MODEL_NAME} ({BEST_MODEL_PATH})"
            )

            bg_imgs = []
            for imgs, _ in EVAL_DS:
                batch = imgs.numpy()
                for row in batch:
                    bg_imgs.append(row)
                    if len(bg_imgs) >= SHAP_MAX_BACKGROUND:
                        break
                if len(bg_imgs) >= SHAP_MAX_BACKGROUND:
                    break
            if len(bg_imgs) < 5:
                raise RuntimeError("insufficient eval images for SHAP background")
            background = np.stack(bg_imgs[:SHAP_MAX_BACKGROUND], axis=0).astype(
                np.float32
            )

            correct_mask = best_y_pred == y_true_all
            correct_idx = np.where(correct_mask)[0]
            incorrect_idx = np.where(~correct_mask)[0]

            chosen_correct = []
            seen_classes = set()
            for i in correct_idx:
                cls = int(y_true_all[i])
                if cls not in seen_classes:
                    chosen_correct.append(int(i))
                    seen_classes.add(cls)
                if len(chosen_correct) >= 3:
                    break
            for i in correct_idx:
                if len(chosen_correct) >= 3:
                    break
                if int(i) not in chosen_correct:
                    chosen_correct.append(int(i))

            chosen = list(chosen_correct[:3])
            if len(incorrect_idx) > 0:
                chosen.append(int(incorrect_idx[0]))
            chosen = chosen[:SHAP_MAX_EXPLAIN]
            chosen_set = set(chosen)

            explain_imgs = []
            explain_titles = []
            cursor = 0
            for imgs, _ in EVAL_DS:
                bs = imgs.shape[0]
                for b in range(bs):
                    g = cursor + b
                    if g in chosen_set:
                        explain_imgs.append(imgs[b].numpy())
                        true_c = int(y_true_all[g])
                        pred_c = int(best_y_pred[g])
                        status = "correct" if pred_c == true_c else "incorrect"
                        explain_titles.append(
                            f"[{status}] true: {index_to_class[true_c]} | pred: {index_to_class[pred_c]}"
                        )
                cursor += bs
                if len(explain_imgs) == len(chosen):
                    break

            if not explain_imgs:
                raise RuntimeError("no explain images selected")

            explain_batch = np.stack(explain_imgs, axis=0).astype(np.float32)
            print(
                f"SHAP setup: background={background.shape[0]} images, "
                f"explain={explain_batch.shape[0]} images"
            )

            explainer = shap.GradientExplainer(
                shap_model, background, batch_size=min(8, SHAP_MAX_BACKGROUND)
            )
            shap_values = explainer.shap_values(explain_batch)

            shap_path = f"{RESULTS_DIR}/shap_examples.png"
            shap.image_plot(
                shap_values,
                explain_batch,
                labels=class_names,
                show=False,
            )
            fig = plt.gcf()
            if explain_titles:
                fig.suptitle(
                    f"SHAP — {BEST_MODEL_NAME} [{EVAL_DS_NAME}] | "
                    + " | ".join(explain_titles[:SHAP_MAX_EXPLAIN]),
                    fontsize=8,
                    y=1.02,
                )
            fig.savefig(shap_path, dpi=150, bbox_inches="tight")
            plt.close(fig)

            SHAP_DONE = True
            print(f"Saved: {shap_path}")
            print(
                "RQ5 SHAP inspection note: inspect whether SHAP attribution regions "
                "(salient red/blue areas in the plot) fall on leaf lesion or tissue "
                "versus background. Grad-CAM is the primary XAI method; SHAP adds "
                "complementary region-attribution evidence — base any report conclusion "
                "on the saved figure, not on assumptions here."
            )

        try:
            if hasattr(signal, "SIGALRM"):
                def _alarm_handler(signum, frame):
                    raise TimeoutError("SHAP explain step exceeded time limit")

                old_handler = signal.signal(signal.SIGALRM, _alarm_handler)
                signal.alarm(SHAP_TIME_LIMIT_SEC)
                try:
                    _run_shap()
                finally:
                    signal.alarm(0)
                    signal.signal(signal.SIGALRM, old_handler)
            else:
                _run_shap()
        except Exception as exc:
            _shap_skip(str(exc))

print(f"SHAP_DONE={SHAP_DONE}")

## 17. Robustness Testing

The best model is evaluated on perturbed versions of the evaluation set (Gaussian noise, blur, brightness, contrast). Perturbations are applied to raw `[0, 255]` images; model preprocessing is in-graph.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

robust_model = tf.keras.models.load_model(BEST_MODEL_PATH, compile=False)
print(f"Loaded best model for robustness testing: {BEST_MODEL_NAME} ({BEST_MODEL_PATH})")

# Materialize raw [0,255] eval images once (uint8 to save memory), unshuffled order aligned with y_true_all.
X_list, y_list = [], []
for imgs, labels in EVAL_DS:
    X_list.append(np.clip(imgs.numpy(), 0, 255).astype(np.uint8))
    y_list.append(np.argmax(labels.numpy(), axis=1))
X_eval = np.concatenate(X_list, axis=0)
y_eval = np.concatenate(y_list, axis=0)

assert y_eval.shape[0] == y_true_all.shape[0], "eval count mismatch vs y_true_all"
assert np.array_equal(y_eval, y_true_all), "y_eval order does not match y_true_all — alignment broken"
print(f"N eval images: {X_eval.shape[0]} | stored as uint8, pixel range [{X_eval.min()}, {X_eval.max()}]")
print("y_eval matches y_true_all — alignment confirmed.")

# Full evaluation set (no subset cap); identical images used for clean + all perturbations.
X_robust = X_eval
y_robust = y_eval
N_ROBUST = int(X_robust.shape[0])
print(f"Using full eval set for robustness: {N_ROBUST} images.")

In [ ]:
import cv2


def add_gaussian_noise(imgs, sigma=15.0):
    rng = np.random.default_rng(SEED)  # deterministic given SEED
    imgs_f = imgs.astype(np.float32)
    noise = rng.normal(0.0, sigma, size=imgs_f.shape).astype(np.float32)
    return np.clip(imgs_f + noise, 0.0, 255.0).astype(np.uint8)


def apply_blur(imgs, ksize=5):
    if ksize % 2 == 0:
        ksize += 1  # cv2 requires odd kernel size
    out = np.empty_like(imgs, dtype=np.uint8)
    for i in range(imgs.shape[0]):
        u8 = imgs[i] if imgs.dtype == np.uint8 else np.clip(imgs[i], 0, 255).astype(np.uint8)
        out[i] = cv2.GaussianBlur(u8, (ksize, ksize), 0)
    return out


def adjust_brightness(imgs, delta=40.0):
    return np.clip(imgs.astype(np.float32) + delta, 0.0, 255.0).astype(np.uint8)


def adjust_contrast(imgs, factor=0.6):
    return np.clip((imgs.astype(np.float32) - 127.5) * factor + 127.5, 0.0, 255.0).astype(np.uint8)

In [ ]:
def evaluate_condition(name, imgs, labels):
    y_pred_parts = []
    n = imgs.shape[0]
    for start in range(0, n, BATCH_SIZE):
        batch_u8 = imgs[start : start + BATCH_SIZE]
        batch_f32 = batch_u8.astype(np.float32)  # model expects raw [0,255] float32
        probs = robust_model.predict(batch_f32, verbose=0)
        y_pred_parts.append(np.argmax(probs, axis=1))
    y_pred = np.concatenate(y_pred_parts, axis=0)
    return {
        "condition": name,
        "accuracy": accuracy_score(labels, y_pred),
        "macro_f1": f1_score(labels, y_pred, average="macro", zero_division=0),
    }


clean_res = evaluate_condition("Clean", X_robust, y_robust)
clean_acc = clean_res["accuracy"]

conditions = [
    ("Gaussian Noise (sigma=15)", add_gaussian_noise(X_robust, sigma=15.0)),
    ("Blur (ksize=5)", apply_blur(X_robust, ksize=5)),
    ("Brightness (+40)", adjust_brightness(X_robust, delta=40.0)),
    ("Contrast (factor=0.6)", adjust_contrast(X_robust, factor=0.6)),
]

rows = [clean_res]
for name, perturbed in conditions:
    rows.append(evaluate_condition(name, perturbed, y_robust))

for r in rows:
    r["delta_vs_clean"] = r["accuracy"] - clean_acc

robustness_df = pd.DataFrame(
    [
        {
            "Condition": r["condition"],
            "N": N_ROBUST,
            "Accuracy": r["accuracy"],
            "Macro F1": r["macro_f1"],
            "Δ Accuracy vs Clean": r["delta_vs_clean"],
        }
        for r in rows
    ]
)
print(f"Robustness results ({BEST_MODEL_NAME}, {EVAL_DS_NAME} split, N={N_ROBUST}):")
print(robustness_df.to_string(index=False))

comparison_csv = f"{RESULTS_DIR}/model_comparison.csv"
if "comparison_df" in globals():
    _comp_ref = comparison_df
elif os.path.isfile(comparison_csv):
    _comp_ref = pd.read_csv(comparison_csv)
else:
    _comp_ref = None

if _comp_ref is not None:
    _eff_rows = _comp_ref[_comp_ref["Model"] == "EfficientNetB0"]
    if len(_eff_rows):
        eff_test_acc = float(_eff_rows.iloc[0]["Test Accuracy"])
        acc_rounded_match = round(clean_acc, 3) == round(eff_test_acc, 3)
        print(
            f"Clean robustness accuracy: {clean_acc:.4f} | "
            f"EfficientNetB0 test accuracy (model_comparison.csv): {eff_test_acc:.4f} | "
            f"match within rounding (3 dp): {acc_rounded_match}"
        )
    else:
        print("NOTE: EfficientNetB0 row not found in model_comparison.csv — skip accuracy cross-check.")
else:
    print("NOTE: model_comparison.csv not available — skip accuracy cross-check.")

robustness_csv = f"{RESULTS_DIR}/robustness_results.csv"
robustness_df.to_csv(robustness_csv, index=False)
print(f"Saved: {robustness_csv}")

### Robustness — Visual Comparison

In [ ]:
# 1) Accuracy per condition (clean highlighted).
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["tab:green"] + ["tab:blue"] * (len(robustness_df) - 1)
bars = ax.bar(robustness_df["Condition"], robustness_df["Accuracy"], color=colors)
ax.set_title(f"Robustness — Accuracy per Condition ({BEST_MODEL_NAME})")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right")
for bar, acc in zip(bars, robustness_df["Accuracy"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{acc:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
acc_path = f"{RESULTS_DIR}/robustness_accuracy.png"
plt.savefig(acc_path, dpi=150)
plt.show()
print(f"Saved: {acc_path}")

# 2) Qualitative grid: one sample image, clean + 4 perturbations.
sample = X_robust[:1]
variants = [
    ("Clean", sample),
    ("Gaussian Noise", add_gaussian_noise(sample, sigma=15.0)),
    ("Blur", apply_blur(sample, ksize=5)),
    ("Brightness +40", adjust_brightness(sample, delta=40.0)),
    ("Contrast 0.6", adjust_contrast(sample, factor=0.6)),
]
fig, axes = plt.subplots(1, len(variants), figsize=(4 * len(variants), 4))
for ax, (title, img) in zip(np.atleast_1d(axes).flatten(), variants):
    ax.imshow(np.clip(img[0], 0, 255).astype(np.uint8))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
ex_path = f"{RESULTS_DIR}/robustness_perturbation_examples.png"
plt.savefig(ex_path, dpi=150)
plt.show()
print(f"Saved: {ex_path}")

In [ ]:
perturbed_only = robustness_df[robustness_df["Condition"] != "Clean"].copy()
most_degraded = perturbed_only.loc[perturbed_only["Δ Accuracy vs Clean"].idxmin()]
least_degraded = perturbed_only.loc[perturbed_only["Δ Accuracy vs Clean"].idxmax()]

print(f"Clean baseline accuracy: {clean_acc:.4f}")
print(
    f"Most degraded: {most_degraded['Condition']} "
    f"(accuracy={most_degraded['Accuracy']:.4f}, Δ={most_degraded['Δ Accuracy vs Clean']:+.4f})"
)
print(
    f"Least degraded: {least_degraded['Condition']} "
    f"(accuracy={least_degraded['Accuracy']:.4f}, Δ={least_degraded['Δ Accuracy vs Clean']:+.4f})"
)

## 18. Artifact Export & Packaging

In [ ]:
import shutil

canonical_best = f"{RESULTS_DIR}/best_model.keras"
shutil.copy(BEST_MODEL_PATH, canonical_best)
print(f"Copied best model ({BEST_MODEL_NAME}) from {BEST_MODEL_PATH} -> {canonical_best}")

# Also copy to repo-root-style path for easy grabbing from the Kaggle Output tab.
shutil.copy(BEST_MODEL_PATH, "/kaggle/working/best_model.keras")
print("Copied best model -> /kaggle/working/best_model.keras")

_preprocessing_map = {
    "Custom CNN": "rescaling_1_255",
    "MobileNetV2": "mobilenet_v2",
    "EfficientNetB0": "efficientnet",
}
preprocessing_type = _preprocessing_map.get(BEST_MODEL_NAME, "rescaling_1_255")

# NOTE: preprocessing is baked INTO the saved model graph, so the Streamlit app must feed
# raw [0,255] images and MUST NOT double-preprocess. This field is for documentation/debugging.
label_map = {
    "num_classes": NUM_CLASSES,
    "img_size": [int(IMG_SIZE[0]), int(IMG_SIZE[1])],
    "best_model": BEST_MODEL_NAME,
    "index_to_class": {str(i): index_to_class[i] for i in range(NUM_CLASSES)},
    "class_to_index": {index_to_class[i]: i for i in range(NUM_CLASSES)},
    "preprocessing": preprocessing_type,
}

for path in (f"{RESULTS_DIR}/label_map.json", "/kaggle/working/label_map.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(label_map, f, indent=2)
    print(f"Saved: {path}")

print("label_map.json contents:")
print(json.dumps(label_map, indent=2))

In [ ]:
import datetime


def _safe(fn, default=None):
    try:
        return fn()
    except Exception:
        return default


run_summary = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "tf_version": tf.__version__,
    "dataset": {
        "data_root": _safe(lambda: DATA_ROOT),
        "layout": _safe(lambda: LAYOUT),
        "num_classes": NUM_CLASSES,
        "class_names": list(class_names),
        "train_count": _safe(lambda: len(train_pairs)),
        "val_count": _safe(lambda: len(val_pairs)),
        "test_count": _safe(lambda: len(test_pairs)),
    },
    "best_val_accuracy": {
        "custom_cnn": _safe(lambda: float(best_val_acc_custom)),
        "mobilenetv2": _safe(lambda: float(best_val_acc_mnv2)),
        "efficientnetb0": _safe(lambda: float(best_val_acc_eff)),
    },
    "eval_split": _safe(lambda: EVAL_DS_NAME),
    "best_model": {
        "name": BEST_MODEL_NAME,
        "test_accuracy": _safe(
            lambda: float(
                comparison_df.loc[comparison_df["Model"] == BEST_MODEL_NAME, "Test Accuracy"].iloc[0]
            )
        ),
        "macro_f1": _safe(
            lambda: float(
                comparison_df.loc[comparison_df["Model"] == BEST_MODEL_NAME, "Macro F1"].iloc[0]
            )
        ),
    },
    "results_files": _safe(lambda: sorted(os.listdir(RESULTS_DIR)), []),
}

run_summary_path = f"{RESULTS_DIR}/run_summary.json"
with open(run_summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)
print("run_summary.json:")
print(json.dumps(run_summary, indent=2))
print(f"Saved: {run_summary_path}")

In [ ]:
EXPECTED_OUTPUTS = [
    "class_distribution.csv",
    "class_distribution.png",
    "sample_images.png",
    "training_curve_custom_cnn.png",
    "training_curve_mobilenetv2.png",
    "training_curve_efficientnetb0.png",
    "confusion_matrix.png",
    "classification_report.csv",
    "model_comparison.csv",
    "robustness_results.csv",
    "gradcam_examples.png",
    "best_model.keras",
    "label_map.json",
]

print("Expected output checklist:")
for fname in EXPECTED_OUTPUTS:
    exists = os.path.exists(f"{RESULTS_DIR}/{fname}")
    print(f"  {'✓' if exists else '✗'} {fname}")

total_bytes = 0
for root, _dirs, files in os.walk(RESULTS_DIR):
    for fn in files:
        total_bytes += os.path.getsize(os.path.join(root, fn))
print(f"\nTotal size of {RESULTS_DIR}: {total_bytes / 1e6:.2f} MB")

In [ ]:
zip_base = "/kaggle/working/results"
zip_path = shutil.make_archive(zip_base, "zip", RESULTS_DIR)
print(f"Created zip: {zip_path} ({os.path.getsize(zip_path) / 1e6:.2f} MB)")

print("\nKey downloadable files (Kaggle Output tab):")
print("  /kaggle/working/results.zip      (everything)")
print("  /kaggle/working/best_model.keras")
print("  /kaggle/working/label_map.json")